# **Scraping Planos Vivo**

_Página de planos do provedor: https://vivo.com.br/para-voce/produtos-e-servicos/para-casa/internet_

## **1 Carga**

In [ ]:
import pandas as pd
df = pd.read_json("../../data/scraped/vivo.json")
df.head()

## **2 Tratamento**

In [ ]:
from datetime import date
import re

# Data e provedor
df["DATA"] = date.today().strftime("%Y-%m-%d")
df["ISP"] = "Vivo"

# Velocidade
df["VELOCIDADE"] = df["textoFibra"].str.extract(r"(\d+)").astype(float)
df.loc[
    df["textoFibra"].str.contains(r"giga|gb|g\b", case=False, na=False, regex=True),
    "VELOCIDADE"
] *= 1000

# SVA: cria uma lista as colunas que iniciam com "cortesiaFibra" ou "sva" ou "textoTv"
# depois concatena essas colunas
cols_sva = [c for c in df if c.startswith(("cortesiaFibra", "sva", "textoTv"))]

df["SVAs"] = df[cols_sva].apply(
    lambda row: ", ".join(row.dropna().astype(str)),
    axis=1
)

# Preço
df["PRECO"] = (
    df["total"]
    .str.replace(r"[^\d,]", "", regex=True)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

#Seleção de colunas
df_tratado = df[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

## **3. Carga**

In [ ]:
# Salvando
provedor = "VIVO.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")